In [0]:
from pyspark.sql import functions as F
bronze_df = spark.read.table("clinical_trials.bronze_studies")

# Bronze Delta Table을 읽는다.
# Bronze에는 ClinicalTrials.gov의 원본 study 구조를 최대한 보존해 두었고,
# Silver에서는 분석과 재사용이 쉬운 형태로 정제·정규화한다.

print(bronze_df.count())
bronze_df.printSchema()

604566
root
 |-- study: struct (nullable = true)
 |    |-- annotationSection: struct (nullable = true)
 |    |    |-- annotationModule: struct (nullable = true)
 |    |    |    |-- unpostedAnnotation: struct (nullable = true)
 |    |    |    |    |-- unpostedEvents: array (nullable = true)
 |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |-- date: string (nullable = true)
 |    |    |    |    |    |    |-- dateUnknown: boolean (nullable = true)
 |    |    |    |    |    |    |-- type: string (nullable = true)
 |    |    |    |    |-- unpostedResponsibleParty: string (nullable = true)
 |    |    |    |-- violationAnnotation: struct (nullable = true)
 |    |    |    |    |-- violationEvents: array (nullable = true)
 |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |-- creationDate: string (nullable = true)
 |    |    |    |    |    |    |-- description: string (nullable = true)
 |    |    |

In [0]:
# ClinicalTrials.gov의 nested JSON에서 임상시험 단위의 핵심 필드를 추출한다.
# 하나의 study가 하나의 row가 되도록 구성해,
# downstream 분석 및 다른 데이터셋과의 join이 쉬운 Silver 테이블을 만든다.

silver_studies_df = bronze_df.select(
    F.col("study.protocolSection.identificationModule.nctId").alias("nct_id"),

    F.col("study.protocolSection.identificationModule.briefTitle").alias("brief_title"),

    F.col("study.protocolSection.identificationModule.officialTitle").alias("official_title"),

    F.col("study.protocolSection.designModule.studyType").alias("study_type"),

    F.col("study.protocolSection.designModule.phases").alias("phases"),

    F.col("study.protocolSection.statusModule.overallStatus").alias("overall_status"),

    F.col("study.protocolSection.designModule.enrollmentInfo.count")
        .cast("long")
        .alias("enrollment_count"),
        # 원본 데이터의 값을 분석 목적에 맞는 데이터 타입으로 변환한다.
# 숫자와 날짜 타입을 명확히 지정해 이후 집계·필터링·날짜 연산을 안정적으로 수행할 수 있게 한다.

    F.col("study.protocolSection.designModule.enrollmentInfo.type")
        .alias("enrollment_type"),

    F.col("study.protocolSection.designModule.designInfo.allocation")
        .alias("allocation"),

    F.col("study.protocolSection.designModule.designInfo.interventionModel")
        .alias("intervention_model"),

    F.col("study.protocolSection.designModule.designInfo.primaryPurpose")
        .alias("primary_purpose"),

    F.col("study.protocolSection.designModule.designInfo.maskingInfo.masking")
        .alias("masking"),

    F.to_date(
        F.col("study.protocolSection.statusModule.startDateStruct.date")
    ).alias("start_date"),

    F.to_date(
        F.col("study.protocolSection.statusModule.completionDateStruct.date")
    ).alias("completion_date"),

    F.to_date(
        F.col("study.protocolSection.statusModule.lastUpdatePostDateStruct.date")
    ).alias("last_update_date"),

    F.col("study.hasResults").alias("has_results"),

    F.col("source"),
    F.col("ingested_at")
)

display(silver_studies_df.limit(20))

# Silver 변환 결과에 기본 데이터 품질 규칙을 적용한다.
# study 단위 테이블이므로 Bronze와 row 수가 동일해야 하며,
# NCT ID는 null이거나 중복되어서는 안 된다.

print("Bronze:", bronze_df.count())
print("Silver:", silver_studies_df.count())

# NCT ID null 확인
silver_studies_df.filter(
    F.col("nct_id").isNull()
).count()

# NCT ID 중복 확인
duplicates_df = (
    silver_studies_df
    .groupBy("nct_id")
    .count()
    .filter(F.col("count") > 1)
)

display(duplicates_df)

nct_id,brief_title,official_title,study_type,phases,overall_status,enrollment_count,enrollment_type,allocation,intervention_model,primary_purpose,masking,start_date,completion_date,last_update_date,has_results,source,ingested_at
NCT00634335,Echocardiographic Characteristics of High Endurance Israeli Athletes,Echocardiographic Characteristics of High Endurance Israeli Athletes,OBSERVATIONAL,null,UNKNOWN,81,ESTIMATED,null,null,null,null,2008-03-01,2009-06-01,2008-12-30,false,clinicaltrials.gov,2026-09-26T12:15:35.981Z
NCT05604235,Effectiveness of Oncological Physiotherapy on Painful Shoulder Cervical Ganglion in Head and Neck Cancer,Effectiveness of Oncological Physiotherapy With Neurodynamics and Kinesitherapy (Active+Passive) on Painful Shoulder Cervical Ganglion Post Cervical Evacuation in Head and Neck Cancer Due to Accessory Spinal Nerve of the Accessory Spinal Nerve,INTERVENTIONAL,List(NA),COMPLETED,20,ACTUAL,RANDOMIZED,PARALLEL,TREATMENT,TRIPLE,2022-11-15,2024-11-15,2024-11-20,false,clinicaltrials.gov,2026-09-26T12:15:35.981Z
NCT06927635,Stereotactic Body Radiotherapy or Standard of Care for Prostate Oligoprogressive Cancer,Stereotactic Body Radiotherapy or Standard of Care for Prostate Oligoprogressive Cancer: a Randomized Phase II Trial,INTERVENTIONAL,List(PHASE2),RECRUITING,75,ESTIMATED,RANDOMIZED,PARALLEL,TREATMENT,NONE,2025-07-08,2030-06-01,2026-07-07,false,clinicaltrials.gov,2026-09-26T12:15:35.981Z
NCT06362135,Virtual Home Visits for Stroke Patients and Their Caregivers,The Evaluation of the Effectiveness of Virtual Home Visits for Stroke Patients and Their Caregivers as Part of Home Health Services.,INTERVENTIONAL,List(NA),UNKNOWN,96,ESTIMATED,RANDOMIZED,PARALLEL,HEALTH_SERVICES_RESEARCH,SINGLE,2023-08-01,2025-03-01,2024-04-15,false,clinicaltrials.gov,2026-09-26T12:15:35.981Z
NCT01805635,PF and FeNO and Clinical Characteristics in Children Being Diagnosed Due to Suspicion of Allergic Diseases,Evaluation of the Relationship Between the Parameters of Pulmonary Function (PF) and the Concentration of Fractional Exhaled Nitric Oxide (FeNO) and Clinical Characteristics in Children Being Diagnosed Due to Suspicion of Allergic Diseases,OBSERVATIONAL,null,UNKNOWN,5500,ESTIMATED,null,null,null,null,2013-02-01,2014-04-01,2013-09-09,false,clinicaltrials.gov,2026-09-26T12:15:35.981Z
NCT01228435,IPI-504 in NSCLC Patients With ALK Translocations,"A Phase II Study of IPI-204, A Novel Hsp90 Inhibitor in NSCLC Patients With ALK Translocations",INTERVENTIONAL,List(PHASE2),TERMINATED,3,ACTUAL,NON_RANDOMIZED,PARALLEL,TREATMENT,NONE,2010-10-01,2012-02-01,2017-12-04,true,clinicaltrials.gov,2026-09-26T12:15:35.981Z
NCT01748435,Pre-emptive Analgesia With Qutenza in Lower Limb Amputation,The Role of Pre-emptive Analgesia With Qutenza (Topical Capsaicin 8%) in Preventing Neuropathic Pain Following Lower Limb Amputation: a Pilot Randomised Controlled Study,OBSERVATIONAL,null,UNKNOWN,30,ESTIMATED,null,null,null,null,2013-02-01,2014-05-01,2012-12-12,false,clinicaltrials.gov,2026-09-26T12:15:35.981Z
NCT04811235,Optical Monitoring With Near-Infrared Spectroscopy for Spinal Cord Injury Trial,"""OMNI-SCI"" Trial - ""Optical Monitoring With Near-Infrared Spectroscopy for Spinal Cord Injury"" Trial",INTERVENTIONAL,List(NA),TERMINATED,1,ACTUAL,NA,SINGLE_GROUP,DEVICE_FEASIBILITY,NONE,2022-09-24,2026-04-01,2026-09-23,false,clinicaltrials.gov,2026-09-26T12:15:35.981Z
NCT04208035,Hepatitis C Elimination in the Netherlands,Hepatitis C Elimination in the Netherlands (CELINE) - A National Multicenter Cohort Study Retrieving Lost to Follow-up Chronic Hepatitis C Patients,OBSERVATIONAL,null,UNKNOWN,250,ESTIMATED,null,null,null,null,2018-09-17,2021-01-01,2019-12-23,false,clinicaltrials.gov,2026-09-26T12:15:35.981Z
NCT04382235,Non-invasive Ventilatory Support of Patients Affected by COVID-19,Management Through Non-invasive Ventilatory Support of Patients Showing an Acute Respiratory Failure Related to COVID-19 in Non-intensive Wards,OBSERVATIONAL,null,COMPLETED

Bronze: 604566
Silver: 604566


nct_id,count


In [0]:
(
    silver_studies_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("clinical_trials.silver_studies")
)

spark.sql("""
SELECT COUNT(*)
FROM clinical_trials.silver_studies
""").show()

+--------+
|COUNT(*)|
+--------+
|  604566|
+--------+



In [0]:
# 한 study에 여러 condition이 존재할 수 있으므로 array를 explode하여
# Study-Condition의 1:N 관계를 별도 Silver 테이블로 정규화한다.
# 이를 통해 질환별 임상시험 수 등의 분석과 join이 쉬워진다.

silver_conditions_df = (
    bronze_df
    .select(
        F.col(
            "study.protocolSection.identificationModule.nctId"
        ).alias("nct_id"),

    # explode_outer : 배열이 없거나 null인 경우도 안전하게 처리합니다.
        F.explode_outer(
            "study.protocolSection.conditionsModule.conditions"
        ).alias("condition")
    )
    .filter(F.col("condition").isNotNull())

    # 실제 condition이 없는 row는 제거
)

display(silver_conditions_df.limit(30))

print("Study rows:", silver_studies_df.count())
print("Condition rows:", silver_conditions_df.count())

nct_id,condition
NCT03705065,Analgesia
NCT05519865,Non Small Cell Lung Cancer
NCT00622765,Obesity
NCT00622765,Nutritional and Metabolic Diseases
NCT00622765,Metabolic Diseases
NCT00622765,Nutrition Disorders
NCT00622765,Overweight
NCT06935565,Cataract
NCT05371067,Gastro-Intestinal Disorder
NCT01525667,Total Hip Arthroplasty


Study rows: 604566
Condition rows: 1087211


In [0]:
# 같은 임상시험에 같은 condition이 두 번 들어가 있다면 탐지
condition_duplicates = (
    silver_conditions_df
    .groupBy("nct_id", "condition")
    .count()
    .filter(F.col("count") > 1)
)

display(condition_duplicates)


# 중복 확인 시, 왜 중복이 생겼는지 확인 후 필요하다면 중복 제거
# silver_conditions_df = silver_conditions_df.dropDuplicates(
#     ["nct_id", "condition"]
# )

nct_id,condition,count


In [0]:
(
    silver_conditions_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("clinical_trials.silver_conditions")
)